<a href="https://www.kaggle.com/code/assadmehboob/heart-failure-prediction?scriptVersionId=308445680" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Heart Failure Prediction:
Cardiovascular diseases (CVDs) are the number 1 cause of death globally, taking an estimated 17.9 million lives each year, which accounts for 31% of all deaths worldwide. Four out of 5CVD deaths are due to heart attacks and strokes, and one-third of these deaths occur prematurely in people under 70 years of age. Heart failure is a common event caused by CVDs and this dataset contains 11 features that can be used to predict a possible heart disease.

People with cardiovascular disease or who are at high cardiovascular risk (due to the presence of one or more risk factors such as hypertension, diabetes, hyperlipidaemia or already established disease) need early detection and management wherein a machine learning model can be of great help.

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder,StandardScaler
le = LabelEncoder()
ss = StandardScaler()
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


## Attribute Information

 ### Age:
 Age of the patient [years]

### Sex:
Sex of the patient [M: Male, F: Female]

### ChestPainType:
Chest pain type [TA: Typical Angina, ATA: Atypical Angina, NAP: Non-
Anginal Pain, ASY: Asymptomatic]

### RestingBP:
Resting blood pressure [mm Hg]

### Cholesterol:
Serum cholesterol [mm/dl]

### FastingBS: 
Fasting blood sugar [1: if FastingBS > 120 mg/dl, 0: otherwise]

### RestingECG:
Resting electrocardiogram results [Normal: Normal, ST: having ST-T wave 
abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV), LVH: 
showing probable or definite left ventricular hypertrophy by Estes' criteria]

### MaxHR:
Maximum heart rate achieved [Numeric value between 60 and 202]

### ExerciseAngina:
Exercise-induced angina [Y: Yes, N: No]

### Oldpeak:
Oldpeak = ST [Numeric value measured in depression]

### ST_Slope:
The slope of the peak exercise ST segment [Up: upsloping, Flat: flat, Down: 
downsloping]

### HeartDisease:
Output class [1: heart disease, 0: Normal]


In [ ]:
df = pd.read_csv("/kaggle/input/heart-failure-prediction/heart.csv")
df.head()

## Data cleaning:

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
def remove_outliers(df, column):
   
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

    num_outliers = len(df) - len(df_filtered)
    print(f"\n--- Outlier Removal Summary for '{column}' ---")
    print(f"Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"Lower Bound: {lower_bound:.2f}, Upper Bound: {upper_bound:.2f}")
    print(f"Original rows: {len(df)}, Filtered rows: {len(df_filtered)}")
    print(f"Number of outliers removed: {num_outliers}")

    return df_filtered

In [ ]:
num_col = df.select_dtypes(include="number")
text_col = df.select_dtypes(include="object")

In [ ]:
num_col.columns

In [ ]:
fig, axes = plt.subplots(3,2, figsize=(10,6))

sns.histplot(data=df, x='Age', kde=True, color="blue", edgecolor="black", ax=axes[0,0])
axes[0,0].set_title("Distribution of Age")

sns.histplot(data=df, x='RestingBP', kde=True, color="blue", edgecolor="black", ax=axes[0,1])
axes[0,1].set_title("Distribution of RestingBP")

sns.histplot(data=df, x='Cholesterol', kde=True, color="blue", edgecolor="black", ax=axes[1,0])
axes[1,0].set_title("Distribution of Cholesterol")

sns.histplot(data=df, x='FastingBS', kde=True, color="blue", edgecolor="black", ax=axes[1,1])
axes[1,1].set_title("Distribution of FastingBS")


sns.histplot(data=df, x="MaxHR", kde=True, color="blue", edgecolor="black", ax=axes[2,0])
axes[2,0].set_title("Distribution of MaxHR")


sns.histplot(data=df, x='Oldpeak', kde=True, color="blue", edgecolor="black", ax=axes[2,1])
axes[2,1].set_title("Distribution of Oldpeak")

plt.tight_layout()
plt.show()

In [ ]:
df = remove_outliers(df,"RestingBP")
df = remove_outliers(df,"Cholesterol")
df = remove_outliers(df,"Oldpeak")



## Exploratory Data Analysis (EDA):

In [ ]:
fig, axes = plt.subplots(3,2, figsize=(10,6))

sns.histplot(data=df, x='Age', kde=True, color="blue", edgecolor="black", ax=axes[0,0])
axes[0,0].set_title("Distribution of Age")

sns.histplot(data=df, x='RestingBP', kde=True, color="blue", edgecolor="black", ax=axes[0,1])
axes[0,1].set_title("Distribution of RestingBP")

sns.histplot(data=df, x='Cholesterol', kde=True, color="blue", edgecolor="black", ax=axes[1,0])
axes[1,0].set_title("Distribution of Cholesterol")

sns.histplot(data=df, x='FastingBS', kde=True, color="blue", edgecolor="black", ax=axes[1,1])
axes[1,1].set_title("Distribution of FastingBS")


sns.histplot(data=df, x="MaxHR", kde=True, color="blue", edgecolor="black", ax=axes[2,0])
axes[2,0].set_title("Distribution of MaxHR")


sns.histplot(data=df, x='Oldpeak', kde=True, color="blue", edgecolor="black", ax=axes[2,1])
axes[2,1].set_title("Distribution of Oldpeak")

plt.tight_layout()
plt.show()

In [ ]:
df.shape

In [ ]:
for text in text_col:
    plt.subplots(figsize=(5,3))
    sns.countplot(data=df, x=text, color="green",edgecolor="grey")
    plt.show()

In [ ]:
for text in text_col:
    plt.Figure(figsize=(8,4))
    df.groupby(text)["HeartDisease"].mean().sort_values(ascending=False).plot(kind="bar")
    plt.show()

In [ ]:
sns.countplot(data=df,x="HeartDisease")
plt.show()

In [ ]:
for text in text_col:
    df[text]=le.fit_transform(df[text])

In [ ]:
df.corr()

## Data Preprocessing:

In [ ]:
df.columns

In [ ]:
x = df[['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS','RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',]]
y = df['HeartDisease']

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
x, y = SMOTE().fit_resample(x,y)

In [ ]:
x = ss.fit_transform(x)

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
y_train.shape

## Model Selection:

### Logistic Regression

In [ ]:
lr=LogisticRegression()
lr.fit(x_train,y_train)
lr_prd = lr.predict(x_test)
lr_prd2 = lr.predict(x_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
accuracy_score(y_test,lr_prd)


In [ ]:
accuracy_score(y_train,lr_prd2)

In [ ]:
pd.DataFrame(classification_report(y_test,lr_prd,output_dict=True))

### SVC:

In [ ]:
svc= SVC()
svc.fit(x_train,y_train)
svc_prd = svc.predict(x_test)
svc_prd2 = svc.predict(x_train)

In [ ]:
print(accuracy_score(y_test,svc_prd))
print(accuracy_score(y_train,svc_prd2))
pd.DataFrame(classification_report(y_test,svc_prd, output_dict=True))

### Decision Tree:

In [ ]:
dtr = DecisionTreeClassifier()
dtr.fit(x_train,y_train)
dtr_prd = dtr.predict(x_test)
dtr_prd2 = dtr.predict(x_train)

In [ ]:
print(accuracy_score(y_test,dtr_prd))
print(accuracy_score(y_train,dtr_prd2))

### naive_bayes:

In [ ]:
from sklearn.naive_bayes import BernoulliNB
nb = BernoulliNB()

In [ ]:
nb.fit(x_train,y_train)
nb_prd = nb.predict(x_test)
nb_prd2 = nb.predict(x_train)

In [ ]:
print(accuracy_score(y_test,nb_prd))
print(accuracy_score(y_train,nb_prd2))
pd.DataFrame(classification_report(y_test,nb_prd,output_dict=True))

###  K Neighbors:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
knc = KNeighborsClassifier()
knc.fit(x_train,y_train)
knc_prd = knc.predict(x_test)
knc_prd2= knc.predict(x_train)

In [ ]:
knc_prm = {
     "n_neighbors":[3,5],
   "p":[1,2],
   "metric":["manhattan","euclidean",'minkowski'],
   
}

In [ ]:
knc_grid = GridSearchCV(knc,knc_prm,cv=5)

In [ ]:
knc_grid.fit(x_train,y_train)

In [ ]:
knc_prm_prd = knc_grid.predict(x_test)
knc_prm_prd2 = knc_grid.predict(x_train)

In [ ]:
print(accuracy_score(y_test,knc_prm_prd))
print(accuracy_score(y_train,knc_prm_prd2))

In [ ]:
knc_grid.best_params_

## conclusion:

Cardiovascular diseases are one of the leading global health issues, accounting for 31% of total deaths worldwide, creating the need for innovative tools and techniques for diagnosis. This study employed a dataset comprising 11 features and selected the machine learning model procedures to help in predictive analysis. Through predictive analysis, the purpose of the model is to help identify high-risk patients that are likely to develop heart problems in the future. The first step taken before modeling was outlier treatment to enhance data quality. After statistical analysis, it was found that more heart disease patients were male. Asymptomatic chest pain is marked as major clinical feature. The most valuable factors included age of the patient, sex, type of chest pain, exercise angina, oldpeak and slope. After constructing the model, the model selection is done based on accuracy. In this step, five models were applied and built to find the best diagnostic tool. The five models included Logistic Regression, SVC, Decision Tree, Naive Bayes and K-Neighbors classifier. The best model was SVC in terms of accuracy with 90%, followed by K-Neighbors with accuracy of 89%. The findings indicate SVC model would facilitate the decision-making process by producing accurate results and advance data-driven clinical decision-making to assist patients with medical management to avert or reduce the likelihood of premature stroke and heart attack.